# 01 · Exploratory Data Analysis: Raw Sensor Data

**Dataset:** Microsoft Azure Predictive Maintenance (Kaggle), with 100 machines and hourly telemetry for 2015.

**Goal of this notebook:** understand the raw data *before* building the pipeline, and answer:

1. What does the data look like (size, time span, quality)?
2. What is the normal operating range of each sensor? Are the validation limits in `config/settings.yaml` sensible?
3. How often do errors, maintenance and failures happen, and on which components / machine models?
4. **Do sensors behave differently in the days before a failure?** If yes, failures are predictable from sensor data, which justifies the ML approach.

**Prerequisites:** run these first so the raw files exist in `data/raw/`:
```
python -m iiot.ingestion.download
python -m iiot.ingestion.validate_raw
```

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from iiot.config import get_settings
from iiot.ingestion.validate_raw import parse_datetimes

settings = get_settings()
RAW = settings.paths.raw
SENSORS = list(settings.sensors)  # volt, rotate, pressure, vibration

# Chart style: validated colour-blind-safe categorical palette (fixed order), recessive grid.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # blue, orange, aqua, yellow
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.color": "#898781", "ytick.color": "#898781", "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE), "lines.linewidth": 2,
    "legend.frameon": False, "legend.fontsize": 8, "figure.dpi": 110,
})
pd.set_option("display.float_format", "{:,.2f}".format)


def load(key: str) -> pd.DataFrame:
    df = pd.read_csv(RAW / settings.dataset.files[key])
    if "datetime" in df.columns:
        df["datetime"] = parse_datetimes(df["datetime"])
    return df


telemetry = load("telemetry")
errors = load("errors")
maint = load("maintenance")
failures = load("failures")
machines = load("machines")
print("Loaded all 5 files from", RAW)

## 1. Dataset overview

In [ ]:
tables = {"telemetry": telemetry, "errors": errors, "maintenance": maint,
          "failures": failures, "machines": machines}
overview = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": ", ".join(df.columns),
        "machines": df["machineID"].nunique(),
        "nulls": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "from": df["datetime"].min() if "datetime" in df else None,
        "to": df["datetime"].max() if "datetime" in df else None,
    }
    for name, df in tables.items()
])
overview

In [ ]:
# Is the telemetry a complete hourly grid? (expected: every machine has a reading every hour)
hours = pd.date_range(telemetry["datetime"].min(), telemetry["datetime"].max(), freq="h")
expected = len(hours) * telemetry["machineID"].nunique()
print(f"Expected hourly readings: {expected:,}")
print(f"Actual readings:          {len(telemetry):,}")
print(f"Coverage:                 {len(telemetry) / expected:.2%}")

## 2. Machines: models and age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

counts = machines["model"].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=PALETTE[0], width=0.6)
axes[0].set_title("Machines per model")
axes[0].set_ylabel("machines")
for x, v in zip(counts.index, counts.values):
    axes[0].annotate(str(v), (x, v), ha="center", va="bottom", fontsize=8, color="#52514e")

axes[1].hist(machines["age"], bins=range(0, machines["age"].max() + 2), color=PALETTE[0],
             edgecolor="#fcfcfb", linewidth=2)
axes[1].set_title("Machine age distribution")
axes[1].set_xlabel("age (years)")
axes[1].set_ylabel("machines")
plt.tight_layout()
plt.show()

machines.groupby("model")["age"].describe()[["count", "mean", "min", "max"]]

## 3. Sensor distributions & validation limits

The Silver layer will treat readings outside the configured limits as sensor glitches.
The limits must sit **well outside** the normal range, so real readings are never discarded.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, sensor, color in zip(axes, SENSORS, PALETTE):
    ax.hist(telemetry[sensor], bins=80, color=color)
    ax.set_title(sensor)
    ax.set_yticks([])
fig.suptitle("Telemetry sensor distributions (all machines, all hours)", fontsize=11)
plt.tight_layout()
plt.show()

limits = pd.DataFrame({s: {"config_min": l.min, "config_max": l.max}
                       for s, l in settings.sensors.items()}).T
stats = telemetry[SENSORS].describe(percentiles=[0.001, 0.5, 0.999]).T
ranges = stats[["min", "0.1%", "50%", "99.9%", "max"]].join(limits)
ranges["limits_ok"] = (ranges["min"] > ranges["config_min"]) & (ranges["max"] < ranges["config_max"])
ranges

## 4. A single machine over time

In [ ]:
# Pick the machine with the most failures and show one month leading up to a failure.
machine_id = failures["machineID"].value_counts().idxmax()
m_fail = failures[failures["machineID"] == machine_id].sort_values("datetime")
end = m_fail["datetime"].iloc[0] + pd.Timedelta(days=3)
start = end - pd.Timedelta(days=30)
m_tel = telemetry[(telemetry["machineID"] == machine_id)
                  & telemetry["datetime"].between(start, end)].set_index("datetime")

fig, axes = plt.subplots(4, 1, figsize=(12, 7.5), sharex=True)
for ax, sensor, color in zip(axes, SENSORS, PALETTE):
    ax.plot(m_tel.index, m_tel[sensor], color=color, linewidth=0.8, alpha=0.45, label="hourly")
    ax.plot(m_tel.index, m_tel[sensor].rolling(24).mean(), color=color, linewidth=2,
            label="24h rolling mean")
    ax.set_ylabel(sensor)
    for _, f in m_fail[m_fail["datetime"].between(start, end)].iterrows():
        ax.axvline(f["datetime"], color="#52514e", linestyle="--", linewidth=1)
axes[0].set_title(f"Machine {machine_id}: 30 days of telemetry (dashed line = failure)")
axes[0].legend(loc="upper left", ncol=2)
plt.tight_layout()
plt.show()

m_fail.head(10)

## 5. Errors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 2]})

counts = errors["errorID"].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=PALETTE[0], width=0.6)
axes[0].set_title("Errors by type")
axes[0].set_ylabel("errors")

monthly = errors.set_index("datetime").resample("MS").size()
axes[1].plot(monthly.index, monthly.values, color=PALETTE[0], marker="o", markersize=5)
axes[1].set_title("Errors per month (all machines)")
axes[1].set_ylim(bottom=0)
plt.tight_layout()
plt.show()

counts.to_frame("errors")

## 6. Maintenance & failures

In [ ]:
components = sorted(set(maint["comp"]) | set(failures["failure"]))
summary = pd.DataFrame({
    "replacements (maintenance)": maint["comp"].value_counts(),
    "failures": failures["failure"].value_counts(),
}).reindex(components).fillna(0).astype(int)
summary["failure_share_%"] = 100 * summary["failures"] / summary["failures"].sum()

fig, ax = plt.subplots(figsize=(7, 3.2))
x = np.arange(len(components))
w = 0.38
ax.bar(x - w / 2 - 0.01, summary["replacements (maintenance)"], w, label="replacements",
       color=PALETTE[0])
ax.bar(x + w / 2 + 0.01, summary["failures"], w, label="failures", color=PALETTE[1])
ax.set_xticks(x, components)
ax.set_title("Component replacements vs failures")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

summary

In [ ]:
# Failure rate per machine model (normalised by the number of machines of each model)
per_model = (failures.merge(machines, on="machineID")
             .groupby("model").size()
             .div(machines["model"].value_counts())
             .rename("failures_per_machine"))
per_machine = failures.groupby("machineID").size().reindex(machines["machineID"], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].bar(per_model.index, per_model.values, color=PALETTE[0], width=0.6)
axes[0].set_title("Failures per machine, by model")
axes[1].hist(per_machine, bins=range(0, per_machine.max() + 2), color=PALETTE[0],
             edgecolor="#fcfcfb", linewidth=2)
axes[1].set_title("Distribution of failures per machine")
axes[1].set_xlabel("failures in 2015")
axes[1].set_ylabel("machines")
plt.tight_layout()
plt.show()

print(f"Machines with no failures: {(per_machine == 0).sum()} of {len(per_machine)}")
per_model.to_frame()

## 7. Key question: do sensors change before a failure?

For every failure we look at the **7 days before it**. Each sensor's 24-hour rolling mean is
standardised per machine (z-score: 0 = that machine's normal level, +1 = one standard deviation above).
If the lines move away from 0 as the failure approaches, the sensor carries an **early warning signal**.

In [ ]:
tel = telemetry.sort_values(["machineID", "datetime"]).set_index("datetime")
rolled = (tel.groupby("machineID")[SENSORS]
          .rolling("24h").mean()
          .reset_index())
# z-score each sensor per machine against that machine's whole-year behaviour
grouped = rolled.groupby("machineID")[SENSORS]
z = (rolled[SENSORS] - grouped.transform("mean")) / grouped.transform("std")
z[["machineID", "datetime"]] = rolled[["machineID", "datetime"]]

HOURS_BEFORE = 168  # 7 days
offsets = pd.DataFrame({"hours_before": np.arange(0, HOURS_BEFORE + 1)})
windows = failures.merge(offsets, how="cross")
windows["datetime"] = windows["datetime"] - pd.to_timedelta(windows["hours_before"], unit="h")
windows = windows.merge(z, on=["machineID", "datetime"], how="inner")

profile = windows.groupby(["failure", "hours_before"])[SENSORS].mean()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=True)
for ax, sensor in zip(axes, SENSORS):
    for comp, color in zip(components, PALETTE):
        if comp in profile.index.get_level_values(0):
            series = profile.loc[comp, sensor]
            ax.plot(-series.index, series.values, color=color, label=comp)
    ax.axhline(0, color="#c3c2b7", linewidth=1)
    ax.set_title(sensor)
    ax.set_xlabel("hours before failure")
axes[0].set_ylabel("z-score (24h mean)")
axes[0].legend(title="failed component", loc="upper left")
fig.suptitle("Average sensor level in the 7 days before a failure, by failed component", fontsize=11)
plt.tight_layout()
plt.show()

# Table view: average z-score in the final 24 hours vs. one week earlier
last_day = windows[windows["hours_before"] <= 24].groupby("failure")[SENSORS].mean()
week_before = windows[windows["hours_before"] >= 144].groupby("failure")[SENSORS].mean()
(last_day - week_before).round(2).rename_axis("failed component").add_suffix(" Δz")

## 8. Do errors increase before a failure?

In [ ]:
# Error rate in the 24h before each failure vs. the average 24h error rate
err_windows = failures.merge(errors, on="machineID", suffixes=("_fail", "_err"))
delta = err_windows["datetime_fail"] - err_windows["datetime_err"]
before = err_windows[(delta > pd.Timedelta(0)) & (delta <= pd.Timedelta(hours=24))]

days = (telemetry["datetime"].max() - telemetry["datetime"].min()) / pd.Timedelta(days=1)
baseline = len(errors) / machines["machineID"].nunique() / days
pre_failure = len(before) / len(failures)
print(f"Average errors per machine per 24h:      {baseline:.3f}")
print(f"Average errors in the 24h before failure: {pre_failure:.3f}")
print(f"Ratio:                                    {pre_failure / baseline:.1f}x")

before.groupby(["failure", "errorID"]).size().unstack(fill_value=0)

## 9. Sensor correlation

In [ ]:
telemetry[SENSORS].corr().round(3)

## 10. Findings

*Fill in after running the notebook on the real data.*

| Question | Finding |
|---|---|
| Data size, time span, completeness | |
| Data quality of the raw source (nulls, duplicates) | |
| Are the sensor validation limits sensible? | |
| Most common errors / failing components | |
| Do some machine models fail more often? | |
| **Which sensors change before which component failures?** | |
| Do errors increase before failures? | |

**Implications for the next phases:**
- *Silver layer:* …
- *Gold layer (features):* e.g. rolling windows long enough to capture the pre-failure drift seen in section 7, error counts per window, time since last maintenance.
- *ML:* …